# Step 6: SmoothQuant 调参 — smoothing_strength（主）+ group_size（次）

**目标**：在 s5 产出的 `s5_tuned`（已定好 ignore 配方）基础上，深入 **SmoothQuant 的灵魂旋钮 `smoothing_strength`(α)**——它控制激活离群点向权重迁移的程度，直接决定 W8A8 精度。α 太小压不住激活离群点（激活量化误差大）、α 太大把误差转嫁给权重（权重量化误差大）——工业上必扫 α 找精度最优。本 step 以 **α 扫描为主**，`group_size`（量化粒度，W8A8 下主要影响 scale 开销）为次要旋钮。L3 读完 s5_tuned 的 ignore 配方、选定最优 α + group_size 后产 **`s6_final`** 到 `out/`，供 s7 四向对比验证。

**算法**（与 s5 跨 step 统一）：`recipe = [SmoothQuantModifier(smoothing_strength=α), GPTQModifier(targets="Linear", scheme="W8A8", ignore=...)]`。

**对应 OUTLINE 课时**：3.6（~30 分钟，原偏 group_size，本次以 SmoothQuant 核心 α 调参为主线）。


## 学完应能讲清（学完本节应能口头回答）

1. `smoothing_strength`(α) 是 SmoothQuant 的**灵魂旋钮**——它数学上做了什么？（按 `s_j = max|a_j|^α / max|w_j|^(1-α)` 算每通道平滑因子，激活除以 s_j、权重乘以 s_j，把激活离群点「搬」到权重侧）
2. α 太小（→0）和 α 太大（→1）分别会怎样？工业上为什么要**扫 α** 找最优？（太小激活离群点压不住、激活量化误差大；太大误差全转嫁给权重、权重量化误差大——两端都不好，中间有最优 α，通常 0.8 左右）
3. α 在 SmoothQuant 里是怎么把「激活难量化」转成「权重好量化」的？为什么这是一个权衡（trade-off）而不是白赚？（激活方差被压低=激活好量化，但代价是权重方差被抬高=权重变难量化，α 调的就是这个迁移比例）
4. `group_size` 在 W8A8 下为什么是**次要**旋钮？（W8A8 是 8bit，group_size 主要影响 scale 存储开销；不像 W4A16 那样对精度敏感）那它主要的两个约束是什么？（必须整除 hidden_size；太小 scale 开销吃压缩比）
5. s6 怎么和 s5 接力？为什么 L3 要先读 s5_tuned 的 ignore 配方再调 α？（ignore 已在 s5 定好「哪些层不量化」，s6 在此基础上调「量化的层怎么平滑」——两者正交，先定层选择再定平滑强度）


In [ ]:
%%capture
import pathlib, os, math
import torch
import torch.nn as nn
import ipytest
try:
    ipytest.autoconfig()
except Exception:
    pass  # nbconvert 非交互上下文（无 IPython shell）
import pytest
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from transformers import Qwen2Config, Qwen2ForCausalLM, AutoTokenizer
from llmcompressor import oneshot
from llmcompressor.modifiers.gptq import GPTQModifier

In [ ]:
# Setup cell（双 env：模块根 = 含 scripts/ + steps/ 的 course/m3-tuning-eval/）。
import pathlib

def _find_module_root(start):
    p = pathlib.Path(start).resolve()
    for cand in [p, *p.parents]:
        if (cand / "scripts").is_dir() and (cand / "steps").is_dir():
            return cand
    raise RuntimeError("找不到模块根（含 scripts/ + steps/ 的目录）")

MODULE_ROOT      = _find_module_root(pathlib.Path.cwd())
MODEL_DIR        = MODULE_ROOT / "models" / "Qwen2.5-7B-Instruct"
TINY_MODEL_DIR   = MODULE_ROOT / "models" / "Qwen2.5-0.5B-Instruct"
OUT_ROOT         = MODULE_ROOT / "out"; OUT_ROOT.mkdir(parents=True, exist_ok=True)
# 接力产物：s6 读 s5_tuned 的 ignore 配方，产 s6_final 供 s7 对比
S5_TUNED_DIR     = OUT_ROOT / "s5_tuned"       # s5 产出（含 s5_tuned_recipe.json）
S6_FINAL_DIR     = OUT_ROOT / "s6_final"       # 本 step 产出（最优 α + group_size）
print("MODULE_ROOT =", MODULE_ROOT, "| 0.5B @", TINY_MODEL_DIR.exists())
print("s5_tuned 接力源 @", S5_TUNED_DIR, "| 存在:", S5_TUNED_DIR.exists())


## 原理：smoothing_strength(α) — SmoothQuant 的灵魂旋钮（OUTLINE 3.6 重构）

> **大前提**：M2 选定 SmoothQuant，s5 定好了 ignore 配方（哪些层回退）。现在调「**量化的层怎么平滑**」——这是 SmoothQuant 特有、其它量化方案（FP8/AWQ）没有的核心旋钮。

**SmoothQuant 的数学**（[论文 2211.10438](https://arxiv.org/abs/2211.10438)）：每个通道 j 算一个平滑因子

```
  s_j = max|a_j|^α  /  max|w_j|^(1-α)        （α ∈ [0,1]）
```

然后把激活除以 s_j、权重乘以 s_j（等价变换，数学上输出不变）：
- **激活侧**：`a_j / s_j` —— 激活离群点被压低，激活变得「好量化」（方差小）。
- **权重侧**：`w_j * s_j` —— 权重方差被抬高，权重变得「难量化」一点。

**α 调的是这个迁移比例**——是一个 trade-off，不是白赚：

| α 值 | 激活侧 | 权重侧 | 结果 |
|---|---|---|---|
| α → 0 | 几乎不平滑（激活离群点原样）| 权重不变 | 激活难量化、误差大（SmoothQuant 没起作用）|
| α = 0.5 | 各担一半 | 各担一半 | 折中 |
| α = 0.8（工业常用）| 激活离群点大幅压低 | 权重略难量化 | 激活好量化、权重还能扛——通常最优 |
| α → 1 | 激活完全压平 | 权重扛全部 | 激活最好量化，但权重误差爆炸 |

**为什么两端都不好、中间有最优**：激活离群点是 INT8 激活量化的主要误差源（OUTLINE 3.1），α 太小压不住；但权重一旦方差被抬太高，W8A8 权重量化（per-channel）也会失精度。工业实践 α 通常在 **0.8 左右**，但不同模型/校准数据有差异——**必扫 α 找精度最优**（s6 L3 真扫）。

**校准数据的影响**（OUTLINE 强化）：α 是在校准数据上估的 `max|a_j|`、`max|w_j|` 算出来的。校准数据分布越接近真实推理分布，α 的迁移越准。校准数据太少/偏移，α 选得再好也救不回。


### group_size（次要旋钮，W8A8 下主要影响 scale 开销）

s5 调的是「哪些层量化/回退」（层选择）；s6 的 group_size 调的是「**每层怎么量化**」（量化粒度）。两者正交。W8A8 下 group_size 对**精度**影响小（8bit 余量大），主要影响 **scale 存储开销**与**合法性约束**：

| group_size | scale 数 | 含义 |
|---|---|---|
| = hidden_size | 1 | per-tensor 退化（合法但精度差）|
| 128（经验最优）| hidden/128 | 分组量化，平衡 |
| 64 | hidden/64 | 更精细（scale 翻倍）|
| 1 | 每行 1 | per-channel 极端 |

**两个硬约束**：
- **必须整除 hidden_size**（否则 vLLM 加载报错，按组切权重对不上）。
- **不是越小越好**：32 以下 scale 存储开销（每 group 一个 scale）占比上升，压缩比反而变差。


## 亲手摸一摸：α 对平滑因子的影响 + group_size 合法性

先用合成数据看 α 怎么改变「激活/权重的相对难度」，再看 hidden_size=4096 下各 group_size 的合法性。


In [ ]:
## 摸一摸：α 对平滑因子 s_j 的影响（合成：max|a|=8 离群, max|w|=2）
max_a, max_w = 8.0, 2.0   # 激活有离群点（8）、权重平稳（2）
alphas = [0.0, 0.3, 0.5, 0.7, 0.8, 0.85, 0.9, 1.0]
print(f"{'α':>5} {'s_j':>8} {'激活 a/s':>10} {'权重 w*s':>10} {'充分平滑?':>10}")
for a in alphas:
    s = (max_a ** a) / (max_w ** (1 - a))
    act_smoothed = max_a / s
    w_scaled = max_w * s
    print(f"{a:>5.2f} {s:>8.3f} {act_smoothed:>10.3f} {w_scaled:>10.3f} {str(act_smoothed <= w_scaled):>10}")
print("\n观察：α↑ → s↑ → 激活被压低（a/s↓）、权重被抬高（w*s↑）。对照上方表格的真实数值：")
print("     α=0 -> s=0.5，激活被「反向放大」到 16.0、权重被压到 1.0（激活非但没被压、反而被抬，相当于 SmoothQuant 完全没起作用、甚至帮倒忙）。")
print("     α=1 -> s=8，激活压到 1.0、权重抬到 16.0（激活全压平、误差全转嫁给权重——权重量化误差爆炸）。")
print("     α≈0.8 -> 激活压到 ~1.74、权重抬到 ~9.19（激活好量化、权重还扛得住——两端都不好、中间有最优）。")
print("     两端都不好：α→0 激活压不住、α→1 权重扛不住，故工业必扫 α 找精度最优（通常 0.8 左右）。")

## 本步填空（2 个：α 主 + group_size 次）

1. **`compare_smoothing_strengths(max_act, max_weight, alphas)`**（**主填空**）—— 扫一组 α，对每个返回平滑代理指标：`s_j`、平滑后激活幅值、平滑后权重幅值、以及**是否充分平滑**（激活幅值是否被压到 ≤ 权重幅值）。**为什么这么设计（填前先想）**：L1/L2 用解析公式（α 的数学定义）算平滑效果代理，验证扫描逻辑；L3 换真 SmoothQuant + 真 PPL 扫描。α 选得对不对，解析代理给方向，真 PPL 给裁决。
2. **`pick_group_size(hidden_size, candidates)`**（**次填空**）—— 从候选 group_size 选拐点：在合法（整除 hidden_size）的候选里，优先选 128（若在候选里）否则选 scale 开销最低（`num_scales/hidden` 最小，即 group_size 最大）的合法项。**为什么这么设计**：W8A8 下 group_size 对精度影响小，决策以**开销 + 合法性**为主（OUTLINE 3.6）。合法性校验（`validate` 的整除逻辑）已归入 ipytest（见下方），本函数只做合法候选的选型决策。


In [ ]:
def compare_smoothing_strengths(max_act, max_weight, alphas):
    """扫一组 α，对每个返回平滑代理指标 dict：
      {alpha, smooth_factor, smoothed_act, smoothed_weight, sufficiently_smoothed}

    SmoothQuant 公式（论文 2211.10438）：
      s_j = max|a|^α / max|w|^(1-α)
      激活除以 s_j（压低离群点）、权重乘以 s_j（吸收）。
    - smooth_factor s = max_act ** alpha / (max_weight ** (1 - alpha))。
    - smoothed_act     = max_act / s（平滑后激活幅值）。
    - smoothed_weight  = max_weight * s（平滑后权重幅值）。
    - sufficiently_smoothed = smoothed_act <= smoothed_weight
      （激活被压到不大于权重——才算充分平滑：INT8 激活量化主要怕激活离群点，
       激活压到权重同量级即可；再压只是把误差转嫁给权重）。

    为什么这么设计（填前先想）：α 调的是激活→权重的误差迁移比例。s 随 α 单调增；
    smoothed_act 随 α 单调减、smoothed_weight 随 α 单调增。找「激活刚被压到 ≤ 权重」的 α
    是解析代理给出的「充分平滑」方向（L3 用真 PPL 精确裁决最优 α）。
    """
    # TODO: 对 alphas 每个 alpha：
    #   1) s = max_act ** alpha / (max_weight ** (1 - alpha))。
    #   2) smoothed_act = max_act / s。
    #   3) smoothed_weight = max_weight * s。
    #   4) sufficiently_smoothed = smoothed_act <= smoothed_weight。
    #   append dict（alpha 用 float(alpha)）。
    #   返回 list（按 alphas 顺序）。
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# L1 测试（compare_smoothing_strengths）——填完立即单独跑此 cell 验证（不依赖 pick_group_size）。

def test_compare_smoothing_strengths_monotone_s():
    # 激活有离群点（max_act=8）、权重平稳（max_weight=2）
    out = compare_smoothing_strengths(8.0, 2.0, [0.0, 0.5, 0.8, 1.0])
    by_a = {d["alpha"]: d for d in out}
    # smooth_factor s 随 α 单调增
    assert by_a[0.0]["smooth_factor"] < by_a[0.5]["smooth_factor"] < by_a[0.8]["smooth_factor"] < by_a[1.0]["smooth_factor"]
    # smoothed_act 随 α 单调减（激活被越压越低）
    acts = [by_a[a]["smoothed_act"] for a in [0.0, 0.5, 0.8, 1.0]]
    assert acts[0] > acts[1] > acts[2] > acts[3], "smoothed_act 应随 α 单调减"
    # smoothed_weight 随 α 单调增（权重越扛越多）
    wts = [by_a[a]["smoothed_weight"] for a in [0.0, 0.5, 0.8, 1.0]]
    assert wts[0] < wts[1] < wts[2] < wts[3], "smoothed_weight 应随 α 单调增"

def test_compare_smoothing_strengths_formula_and_conservation():
    # 等价变换：smoothed_act * smoothed_weight == max_act * max_weight（守恒，输出不变）
    out = compare_smoothing_strengths(8.0, 2.0, [0.3, 0.8])
    for d in out:
        assert abs(d["smoothed_act"] * d["smoothed_weight"] - 8.0 * 2.0) < 1e-9, "等价变换应守恒"
        # 公式核对：s = max_act^a / max_w^(1-a)
        a = d["alpha"]
        s_expected = (8.0 ** a) / (2.0 ** (1 - a))
        assert abs(d["smooth_factor"] - s_expected) < 1e-9
        assert abs(d["smoothed_act"] - 8.0 / s_expected) < 1e-9
        assert d["sufficiently_smoothed"] == (d["smoothed_act"] <= d["smoothed_weight"])

def test_compare_smoothing_strengths_endpoints():
    out = compare_smoothing_strengths(8.0, 2.0, [0.0, 1.0])
    by_a = {d["alpha"]: d for d in out}
    # α=0：s = max_act^0/max_w^1 = 1/2；smoothed_act = 8/0.5 = 16（？）— 不，s=1/2 则 act=8/(1/2)=16，weight=2*(1/2)=1
    # 重新核：α=0 -> s = 8^0 / 2^1 = 1/2 -> act=8/0.5=16, w=2*0.5=1（激活没压反而被放，没平滑）
    assert abs(by_a[0.0]["smooth_factor"] - 0.5) < 1e-9
    assert abs(by_a[0.0]["smoothed_act"] - 16.0) < 1e-9
    assert by_a[0.0]["sufficiently_smoothed"] is False  # 激活 16 > 权重 1，没平滑
    # α=1：s = 8^1 / 2^0 = 8；act=8/8=1, w=2*8=16（激活全压平、权重全扛）
    assert abs(by_a[1.0]["smooth_factor"] - 8.0) < 1e-9
    assert by_a[1.0]["sufficiently_smoothed"] is True   # 激活 1 <= 权重 16


In [ ]:
def pick_group_size(hidden_size, candidates):
    """从候选 group_size 选拐点（W8A8 下决策以开销+合法性为主）。
    1) 只保留合法候选：1 <= gs <= hidden_size 且 hidden_size % gs == 0。
    2) 若 128 在合法候选里，优先返回 128（OUTLINE 经验最优）。
    3) 否则返回 scale 开销最低的合法候选——开销 = num_scales/hidden_size，
       num_scales = hidden_size // gs，开销最低 = num_scales 最小 = gs 最大。
    4) 若无任何合法候选，raise ValueError。

    返回选中的 group_size（int）。

    为什么这么设计（填前先想）：W8A8 下 group_size 对精度影响小（8bit 余量大），
    决策主要看 scale 开销（小=省显存/省压缩比损失）+ 合法性。128 是工业经验最优（scale 数与
    精度的平衡点）；没 128 就选开销最低的合法项（最大 group_size）。合法性本身的校验
    （整除/正数/<=hidden）已归入下方 ipytest 的 validate 行为断言，本函数只做选型决策。
    """
    # TODO:
    #   1) legal = [gs for gs in candidates if isinstance(gs,int) and 1 <= gs <= hidden_size and hidden_size % gs == 0]
    #   2) if not legal: raise ValueError("无合法 group_size 候选")
    #   3) if 128 in legal: return 128
    #   4) return max(legal)  （开销最低 = group_size 最大）
    raise NotImplementedError


In [ ]:
%%ipytest -qq
# L1 测试（pick_group_size）+ 归测试的 validate 行为（整除/边界校验进断言，不占填空）。

def test_pick_group_size_prefers_128():
    assert pick_group_size(4096, [32, 64, 128, 256]) == 128, "128 在候选里应优先"
    assert pick_group_size(4096, [64, 128, 256, 170]) == 128  # 170 非法被剔除，128 仍优先

def test_pick_group_size_no_128_picks_lowest_overhead():
    # 候选无 128：选开销最低 = group_size 最大
    assert pick_group_size(4096, [64, 256]) == 256, "无 128 时选开销最低（最大 group_size）"
    assert pick_group_size(4096, [32, 64]) == 64

def test_pick_group_size_all_illegal_raises():
    with pytest.raises(ValueError, match="无合法"):
        pick_group_size(4096, [170, 300, 500])  # 都不整除

# === 归测试：原 validate_group_size 的整除/边界逻辑（spec §4 s6：validate 归测试）===
# 这部分是「薄校验」，不占填空——直接用断言固化合法性判据。pick_group_size 内部也用同一判据。
def _is_legal_group_size(group_size, hidden_size):
    return (isinstance(group_size, int) and group_size >= 1
            and group_size <= hidden_size and hidden_size % group_size == 0)

def test_group_size_legality_judgement():
    # 整除判据：合法因子的集合
    assert _is_legal_group_size(128, 4096) is True
    assert _is_legal_group_size(64, 4096) is True
    assert _is_legal_group_size(1, 4096) is True       # per-channel 极端
    assert _is_legal_group_size(4096, 4096) is True    # per-tensor 退化
    # 非法：不整除
    assert _is_legal_group_size(170, 4096) is False, "170 不整除 4096"
    assert _is_legal_group_size(300, 4096) is False
    # 非法：越界
    assert _is_legal_group_size(8192, 4096) is False, "group_size 不能大于 hidden_size"
    assert _is_legal_group_size(0, 4096) is False, "group_size 必须 >= 1"
    assert _is_legal_group_size(-128, 4096) is False


## L2（CPU）：α 扫描代理 + group_size 选型决策

用 `compare_smoothing_strengths` 跑 α 扫描（合成激活/权重幅值），画「平滑后激活 vs 权重」随 α 变化曲线，找到「充分平滑」的 α 范围。再用 `pick_group_size` 在 tiny hidden_size=64 上做选型决策。验证扫描逻辑 + 选型决策在合理范围内。


In [ ]:
## L2：α 扫描代理（合成激活 max=8, 权重 max=2）
max_a, max_w = 8.0, 2.0
alphas = [0.0, 0.3, 0.5, 0.7, 0.8, 0.85, 0.9, 1.0]
rows = compare_smoothing_strengths(max_a, max_w, alphas)
print(f"α 扫描（max_act={max_a}, max_weight={max_w}）：")
print(f"{'α':>5} {'s':>8} {'act/s':>8} {'w*s':>8} {'充分?':>6}")
for r in rows:
    print(f"{r['alpha']:>5.2f} {r['smooth_factor']:>8.3f} {r['smoothed_act']:>8.3f} {r['smoothed_weight']:>8.3f} {str(r['sufficiently_smoothed']):>6}")

fig, ax = plt.subplots(figsize=(7,3.5))
ax.plot([r['alpha'] for r in rows], [r['smoothed_act'] for r in rows], 'o-', label='smoothed act (压低)')
ax.plot([r['alpha'] for r in rows], [r['smoothed_weight'] for r in rows], 's-', label='smoothed weight (抬高)')
# 标充分平滑区（act <= weight）
ss = [r['alpha'] for r in rows if r['sufficiently_smoothed']]
if ss:
    ax.axvspan(ss[0], ss[-1], alpha=0.15, color='green', label='充分平滑区')
ax.set_xlabel('smoothing_strength α'); ax.set_ylabel('幅值'); ax.set_title('L2 α 扫描：激活压低 vs 权重抬高')
ax.legend(); fig.tight_layout(); plt.show()

print(f"\n充分平滑的 α 范围：{ss if ss else '无（激活离群太狠）'} → 工业通常在此范围内用真 PPL 精确裁决（L3）")
print("注意：充分平滑（act ≤ weight）范围较宽 [0.5..1.0]，但「充分」≠「最优」——")
print("  α=0.5 激活刚被压到权重同量级，权重量化误差已经开始抬升；")
print("  α≈0.8 是工业经验最优（激活好量化、权重还扛得住，PPL 通常最低）；")
print("  最终裁决靠 L3 真 PPL 在此范围内扫描——代理给方向、真 PPL 给数值。")
assert ss, "至少应有一个 α 达到充分平滑"

# group_size 选型决策（tiny hidden=64）
gs = pick_group_size(64, [1, 2, 4, 8, 16, 32, 64])
print(f"\npick_group_size(64, [...]) = {gs}（128 不在候选，选开销最低的最大合法项）")
print("L2 通过：α 扫描代理单调性正确 + 充分平滑范围识别 + group_size 选型决策合理。")

## L3（H200，GPU + SKIP_L3 双守卫）：真 7B α 扫描 + group_size + 产 s6_final

在 7B 上**真扫 α**（如 0.0/0.5/0.8/0.85/1.0）找 W8A8 精度最优，验证 α 太小/太大的后果。**先读 s5_tuned 的 ignore 配方**（接力），在其基础上扫 α + group_size，选最优组合产 **`s6_final`** 到 `out/`。L3 实测 PPL（不只磁盘）——这是 SmoothQuant 工业调参的核心实证。

> 缺 s5_tuned 产物时友好降级（用 `ignore=["lm_head"]` 兜底，打印 warning）。

> L3 双守卫：除 GPU 外，reviewer 执行验证设 `SKIP_L3=1` 跳过（扫 α 真重量化慢，只验 L1+L2 代码逻辑）；真人/学员跑不设，L3 实证。


In [ ]:
import torch, os, json, shutil
def run_l3_alpha_scan():
    from llmcompressor.modifiers.transform.smoothquant import SmoothQuantModifier
    from datasets import load_dataset
    from transformers import AutoModelForCausalLM
    calib = load_dataset("wikitext", "wikitext-2-raw-v1", split="train").shuffle(seed=0)["text"][:128]

    # === 接力：读 s5_tuned 的 ignore 配方 ===
    ignore = ["lm_head"]  # 兜底
    recipe_path = S5_TUNED_DIR / "s5_tuned_recipe.json"
    if recipe_path.exists():
        ignore = json.loads(recipe_path.read_text())["ignore"]
        print(f"[接力] 读 s5_tuned ignore 配方：{len(ignore)} 层")
    else:
        print(f"[warn] 无 s5_tuned 产物（{recipe_path}），用 ignore=['lm_head'] 兜底")

    hidden = json.loads((MODEL_DIR / 'config.json').read_text())['hidden_size']
    group_size = pick_group_size(hidden, [64, 128, 256])  # 次要旋钮：选型决策
    print(f"group_size 选定 = {group_size}（hidden_size={hidden}）")

    def ppl_of(model_dir):
        m = AutoModelForCausalLM.from_pretrained(model_dir, torch_dtype=torch.float16, device_map="auto")
        tok = AutoTokenizer.from_pretrained(model_dir)
        ids = tok("The future of AI depends on efficient inference at scale.", return_tensors="pt").input_ids.to(m.device)
        with torch.no_grad(): logits = m(ids).logits[0]
        loss = torch.nn.functional.cross_entropy(logits[:-1], ids[0,1:])
        return float(torch.exp(loss).item())

    # === 主旋钮：扫 α（SmoothQuant 灵魂参数）===
    alphas = [0.0, 0.5, 0.8, 0.85, 1.0]
    scan = []
    for a in alphas:
        recipe = [SmoothQuantModifier(smoothing_strength=a),
                  GPTQModifier(targets="Linear", scheme="W8A8", ignore=ignore)]
        out = OUT_ROOT / f"s6_alpha{a}"
        oneshot(model=str(MODEL_DIR), tokenizer=str(MODEL_DIR), recipe=recipe,
                dataset=calib, num_calibration_samples=128, output_dir=str(out))
        ppl = ppl_of(out)
        scan.append({"alpha": a, "ppl": ppl})
        print(f"  α={a:.2f} -> PPL={ppl:.3f}")

    best = min(scan, key=lambda r: r["ppl"])
    print(f"\n最优 α = {best['alpha']}（PPL={best['ppl']:.3f}）—— 验证 α 太小/太大都不好、中间最优")
    json.dump({"alpha_scan": scan, "best_alpha": best["alpha"],
               "group_size": group_size, "ignore": ignore},
              open(OUT_ROOT / "s6_alpha_scan.json", "w"), indent=2)

    # === 闭环产物：s6_final（最优 α + group_size）===
    final_src = OUT_ROOT / f"s6_alpha{best['alpha']}"
    if final_src.exists():
        if S6_FINAL_DIR.exists(): shutil.rmtree(S6_FINAL_DIR)
        shutil.copytree(final_src, S6_FINAL_DIR)
        json.dump({"alpha": best["alpha"], "group_size": group_size,
                   "ignore": ignore, "source_step": "s6", "scheme": "W8A8"},
                  open(S6_FINAL_DIR / "s6_final_recipe.json", "w"), indent=2)
        print(f"[产物] s6_final @ {S6_FINAL_DIR}（最优 α={best['alpha']}, group_size={group_size}）")

if torch.cuda.is_available() and not os.environ.get("SKIP_L3"):
    run_l3_alpha_scan()
else:
    print("跳过 L3：无 GPU 或 SKIP_L3=1（CPU/CI 只验 L1+L2 α 扫描/group_size 选型逻辑）")


## 产物检查：α 扫描曲线 + s6_final

L3 跑完会在 `out/` 产 `s6_alpha_scan.json`（α→PPL 扫描）+ `s6_final/`（最优 α + group_size 模型）。s6_final 是 s7 四向对比的「调优终点」产物。


In [ ]:
import json, pathlib
p = OUT_ROOT / "s6_alpha_scan.json"
if p.exists():
    d = json.loads(p.read_text())
    scan = d["alpha_scan"]
    print("真 7B α 扫描（PPL 越低越好）：")
    for r in scan: print(f"  α={r['alpha']:.2f}  PPL={r['ppl']:.3f}")
    print(f"最优 α = {d['best_alpha']}（验证两端差、中间优）")
    fig, ax = plt.subplots(figsize=(7,3.5))
    ax.plot([r['alpha'] for r in scan], [r['ppl'] for r in scan], 'o-')
    ax.axvline(d['best_alpha'], ls=':', color='red', label=f"最优 α={d['best_alpha']}")
    ax.set_xlabel('smoothing_strength α'); ax.set_ylabel('PPL'); ax.set_title('7B α 扫描（SmoothQuant 调参）'); ax.legend()
    fig.tight_layout(); plt.show()
else:
    print(f"{p} 不存在（L3 未跑或被 SKIP_L3 跳过）")

print("\n=== 闭环产物接力状态 ===")
for name, d in [("s6_final（最优 α+group_size，s7 读）", S6_FINAL_DIR)]:
    status = f"@ {d}" if d.exists() else "未生成（跑 L3 生成，s7 会友好降级）"
    print(f"  {name}: {status}")
